# Pitcher Data
This notebook is intended to create the `pitchers.csv` file that the main project uses.

In [1]:

import time
import pandas as pd
from pybaseball import batting_stats_bref, chadwick_register, cache

cache.enable()  # caches downloads so reruns are fast

Get every batter per season in the range 2016-2026

In [2]:
START, END = 2016, 2026

frames = []
for season in range(START, END + 1):
    try:
        df = batting_stats_bref(season)
        df = df[["mlbID", "Name"]].copy()
        df["season"] = season
        frames.append(df)
        print(f"{season}: {len(df)} batters")
    except Exception as e:
        print(f"{season}: failed -> {e}")
    time.sleep(3)  # be polite to Baseball Reference

seasons = pd.concat(frames, ignore_index=True)
seasons["mlbID"] = pd.to_numeric(seasons["mlbID"], errors="coerce")
seasons = seasons.dropna(subset=["mlbID"]).astype({"mlbID": int})

# one row per pitcher, with the seasons they pitched in the window
window = (
    seasons.groupby("mlbID")
    .agg(
        window_first_season=("season", "min"),
        window_last_season=("season", "max"),
        seasons_batted=("season", "nunique"),
    )
    .reset_index()
)
print(len(window), "unique pitchers")

2016: 970 batters
2017: 959 batters
2018: 993 batters
2019: 991 batters
2020: 588 batters
2021: 1055 batters
2022: 699 batters
2023: 660 batters
2024: 654 batters
2025: 675 batters
2026: 664 batters
2634 unique pitchers


Join to the Chadwick Reigster (register of all professional baseball players)

In [3]:
reg = chadwick_register()

reg = reg[
    [
        "key_mlbam", "key_bbref", "key_fangraphs", "key_retro",
        "name_first", "name_last",
        "mlb_played_first", "mlb_played_last",
    ]
].copy()
reg["key_mlbam"] = pd.to_numeric(reg["key_mlbam"], errors="coerce")
reg = reg.dropna(subset=["key_mlbam"]).astype({"key_mlbam": int})

batters = window.merge(reg, left_on="mlbID", right_on="key_mlbam", how="left")

batters["name"] = (batters["name_first"].fillna("") + " " + batters["name_last"].fillna("")).str.strip()

batters = batters.rename(columns={
    "mlbID": "mlbam_id",
    "key_bbref": "bbref_id",
    "key_fangraphs": "fangraphs_id",
    "key_retro": "retrosheet_id",
    "mlb_played_first": "career_start",
    "mlb_played_last": "career_end",
})

# blank career_end for players whose last season is the latest in the data (presumably active)
batters.loc[batters["career_end"] >= END, "career_end"] = pd.NA

# fall back to the window seasons where the register has no dates
batters["career_start"] = batters["career_start"].fillna(batters["window_first_season"])

cols = [
    "mlbam_id", "bbref_id", "fangraphs_id", "retrosheet_id", "name",
    "career_start", "career_end",
    "window_first_season", "window_last_season","seasons_batted"
]
batters = batters[cols].sort_values("name").reset_index(drop=True)
batters.to_csv("batters.csv", index=False)

print(batters.shape)
batters.head(10)

(2634, 10)


,mlbam_id,bbref_id,fangraphs_id,retrosheet_id,name,career_start,career_end,window_first_season,window_last_season,seasons_batted
0,595918,coleaj01,11467,colea002,A. J. Cole,2015.0,2021.0,2016,2018,3
1,454560,ellisaj01,5677,ellia001,A. J. Ellis,2008.0,2018.0,2016,2018,3
2,805999,ewingaj01,-1,ewina001,A. J. Ewing,2026.0,NaN,2026,2026,1
3,456167,griffaj01,11132,grifa002,A. J. Griffin,2012.0,2017.0,2016,2017,2
4,543362,jimenaj01,7077,jimea001,A. J. Jiménez,2017.0,2017.0,2017,2017,1
5,621345,minteaj01,18655,minta001,A. J. Minter,2017.0,NaN,2021,2021,1
6,571963,morriaj01,9919,morra002,A. J. Morris,2016.0,2016.0,2016,2016,1
7,150229,pierza.01,746,piera001,A. J. Pierzynski,1998.0,2016.0,2016,2016,1
8,607223,reedaj01,16246,reeda002,A. J. Reed,2016.0,2019.0,2016,2019,4
9,519263,schugaj01,11432,schua001,A. J. Schugel,2015.0,2017.0,2016,2017,2


Sanity Check

In [4]:
print("duplicate ids:", batters["mlbam_id"].duplicated().sum())
print("missing names:", (batters["name"] == "").sum())
print("missing bbref ids:", batters["bbref_id"].isna().sum())
print("active (no career_end):", batters["career_end"].isna().sum())

duplicate ids: 0
missing names: 0
missing bbref ids: 0
active (no career_end): 875
